In [ ]:
import sys 
sys.path.append(os.getcwd())
from pathlib import Path
import os
import pandas as pd 
import numpy as np
# ==========================================
# 1. CORE DIRECTORIES
# ==========================================
ONEDRIVE_ROOT = Path(os.environ.get('OneDrive', ''))
DATA_ROOT = ONEDRIVE_ROOT / '0. DATASETS'
RAW_DATA_DIR = DATA_ROOT / 'raw'
OUTPUT_DIR = DATA_ROOT / 'outputs'
PROJECT_ROOT = OUTPUT_DIR / 'Direct-Investing-TLH'

# ==========================================
# 2. PIPELINE SUBDIRECTORIES
# ==========================================
# Creating the dedicated output folders for each stage of the new engine
DATA_DIR = PROJECT_ROOT / '01_Data'


In [ ]:
# Loading our daily market data 
df_univ = pd.read_parquet(DATA_DIR / 'tlh_universe.parquet')
# choosing the period
start_date = pd.to_datetime('2015-01-01')
end_date = pd.to_datetime ('2023-12-31')
df_univ = df_univ[(df_univ['date'] >= start_date) & 
                  (df_univ['date'] <= end_date)]
df_univ.set_index('date', inplace=True)
df_univ

,permno,siccd,dlyret,dlyretx,sprtrn,currency,prc_adj_usd,shrout_adj,mkt_cap_usd,divamt_net_usd,usd_cad,prc_adj_cad,divamt_net_cad
date,,,,,,,,,,,,,
2015-01-02,10104,7372.0,-0.014232,-0.014232,-0.000340,USD,44.33,4391367.0,1.946693e+08,0.000,1.1725,51.976925,0.000000
2015-01-05,10104,7372.0,-0.013986,-0.016693,-0.018278,USD,43.59,4391367.0,1.914197e+08,0.102,1.1785,51.370815,0.120207
2015-01-06,10104,7372.0,-0.010323,-0.010323,-0.008893,USD,43.14,4391367.0,1.894436e+08,0.000,1.1802,50.913828,0.000000
2015-01-07,10104,7372.0,0.000232,0.000232,0.011630,USD,43.15,4391367.0,1.894875e+08,0.000,1.1851,51.137065,0.000000
2015-01-08,10104,7372.0,0.006025,0.006025,0.017888,USD,43.41,4391367.0,1.906292e+08,0.000,1.1812,51.275892,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,93436,3711.0,-0.007701,-0.007701,0.001660,USD,252.54,3178921.0,8.028047e+08,0.000,1.3259,334.842786,0.000000
2023-12-26,93436,3711.0,0.016116,0.016116,0.004232,USD,256.61,3178921.0,8.157429e+08,0.000,1.3208,338.930488,0.000000
2023-12-27,93436,3711.0,0.018822,0.018822,0.001430,USD,261.44,3178921.0,8.310971e+08,0.000,1.3201,345.126944,0.000000


In [ ]:
trading_days = df_univ.index.unique().sort_values()
# first day of trading
day_1 = trading_days[0]
# market data on day 1
daily_data_1 = df_univ.loc[day_1]

benchmark_permnos = daily_data_1['permno'].values

# dictionaries are much faster than .loc on series or dfs. 
# creating dictionaries for prices and dividends 
prices_cad = dict(zip(daily_data_1['permno'], daily_data_1['prc_adj_cad']))
divs_cad = dict(zip(daily_data_1['permno'], daily_data_1['divamt_net_cad']))7

total_mkt_cap = daily_data_1['mkt_cap_usd'].sum()

# benchmark weights 
weights = daily_data_1['mkt_cap_usd'] / total_mkt_cap
bench_w = pd.Series(weights.values, index = daily_data_1['permno'])

In [ ]:
bench_w

permno
10104    0.010332
10107    0.020395
10138    0.001193
10145    0.004164
10147    0.003222
           ...   
92988    0.000644
93002    0.001356
93096    0.001140
93159    0.000375
93422    0.000191
Length: 502, dtype: float64

In [ ]:


# CRA 30-Day Superficial Loss Trackers
# when we sell a stock, we are going to record the date of the sale so that we don't by back the stock
superficial_loss_lockouts = {}  # Forward rule: {permno: expiration_date}

initial_capital_cad = 1_000_000
# recording cash 
# initial cash (cash at time 0 ) is initial capital 
cash_cad = initial_capital_cad

# DAY 1: INITIALIZATION (Buying the Benchmark)
print(f"--- Processing Day 1: {day_1.date()} ---")

# 1. We want to perfectly track the benchmark on Day 1.
# Target dollars = benchmark weight * initial cash
# target_shares = {permno: shares}

target_shares = {}
for permno, weight in bench_w.items():
    # failsafe
    if pd.isna(weight) or weight <= 0:
        continue
    # get current price 
    price = prices_cad[permno]
    
    # failsafe: Skip if price is missing, NaN, or zero
    if price is None or pd.isna(price) or price <= 0:
        
        continue

    target_dollars = weight * initial_capital_cad
    # Truncate to 4 decimals for Wealthsimple fractional shares
    raw_shares = target_dollars / price
    target_shares[permno] = np.floor(raw_shares * 10000.0) / 10000.0

--- Processing Day 1: 2015-01-02 ---


In [217]:
dict(list(target_shares.items())[:5])

{10104: 198.7767,
 10107: 371.9906,
 10138: 11.8628,
 10145: 35.4341,
 10147: 92.1108}

In [ ]:
# EXECUTING BUYS ON DAY 1
trade_history =[]               # Audit Trail
last_buy_dates = {}             # Recording for Backward rule: {permno: last_purchase_date}
do_not_harvest = []             # a list of permnos not to harvest because they were bought within the last 30 days
positions = {}                  # inititate our portfolio (empty on day zero)

# mark-to-Market : initiate value of the portfolio (0 on day 0)
total_equity_cad_1 = 0.0

# 2. Execute the "Buys" to establish the ACB and deplete Cash
for permno, shares in target_shares.items():
    # failsafe 
    if pd.isna(shares) or shares <= 0:
        continue
    price = prices_cad[permno]
    
    #failsafe
    if price is None or pd.isna(price) or price <= 0:
        continue

    cost = shares * price
    # add to aum
    total_equity_cad_1 += cost 
    # Deduct cash
    cash_cad -= cost
    # Record the position and the ACB
    positions[permno] = {
        'shares': shares, 
        'acb_per_share': price  # The ACB is locked in on Day 1
    }
    
    # Log the 30-Day Backward Rule (Cannot harvest if bought in last 30 days)
    last_buy_dates[permno] = day_1
total_aum_1 = total_equity_cad_1 + cash_cad

do_not_harvest =[
    permno for permno, last_buy in last_buy_dates.items()
    if (day_1 - last_buy).days <= 30
]

print(f"Number of positions established: {len(positions)}")
print(f"Total AUM @ the End of Day 1:  ${total_aum_1}")
print(f"End of Day 1 Cash: ${cash_cad:,.2f}")
print(f"Number of stocks we are not allowed to harvest @ the end of Day 1 : {len(do_not_harvest)}")


Number of positions established: 502
Total AUM @ the End of Day 1:  $1000000.0000000001
End of Day 1 Cash: $2.50
Number of stocks we are not allowed to harvest @ the end of Day 1 : 502


In [219]:
# sample positions 
dict(list(positions.items())[:5])

{10104: {'shares': 198.7767, 'acb_per_share': 51.976925},
 10107: {'shares': 371.9906, 'acb_per_share': 54.826100000000004},
 10138: {'shares': 11.8628, 'acb_per_share': 100.55360000000002},
 10145: {'shares': 35.4341, 'acb_per_share': 117.51967500000002},
 10147: {'shares': 92.1108, 'acb_per_share': 34.975675}}

In [220]:
# DAY 2: THE MARKET MOVES (Scanning for Losses)
# Advance the clock to the next trading day
# Calculate AUM before possible rebalancing  
day_2 = trading_days[1]
print(f"\n--- Processing Day 2: {day_2.date()} ---")

# 1. Get the new state of the world
daily_data_2 = df_univ.loc[day_2].copy()
daily_data_2 = daily_data_2.replace([np.inf, -np.inf], np.nan)

prices_cad_2 = dict(zip(daily_data_2['permno'], daily_data_2['prc_adj_cad']))

# 2. Mark-to-Market (Calculate new Total AUM based on Day 2 prices)
total_equity_cad_2 = 0.0

# going over every position from the previous period 
for permno, pos in positions.items():

    shares = pos['shares']
    
    if pd.isna(shares) or shares is None: 
        continue 
    # Failsafe: if price is missing, use ACB 
    price_today = prices_cad_2.get(permno, pos['acb_per_share']) 
    total_equity_cad_2 += shares * price_today



--- Processing Day 2: 2015-01-05 ---


In [221]:
#========================
# DIVIDENDS 
divs_cad_2 = dict(zip(daily_data_2['permno'], daily_data_2['divamt_net_cad']))
# initialize day 40 dividends 
daily_dividend_income = 0.0
# Loop through what we own and see if they paid us today
for permno, pos in positions.items():
    dps = divs_cad_2.get(permno, 0.0)
    if dps > 0:
        payout = pos['shares'] * dps
        cash_cad += payout  # Add to our global cash balance!
        daily_dividend_income += payout

if daily_dividend_income > 0:
    print(f"Collected ${daily_dividend_income:,.2f} in CAD dividends on {day_2.date()}.")

Collected $58.32 in CAD dividends on 2015-01-05.


In [222]:
total_aum_2 = total_equity_cad_2 + cash_cad
portfolio_return_2 = (total_aum_2 - total_aum_1) / total_aum_1
print(f"Day 2 Total AUM: ${total_aum_2:,.2f}")
print(f"Portfolio Return @ Day 2: ${portfolio_return_2:,.4f}")


Day 2 Total AUM: $986,902.82
Portfolio Return @ Day 2: $-0.0131


In [223]:
# 3. THE HARVEST SCANNER
# We loop through our positions and check if anything dropped below our threshold (-5%) 
# and collect them in harvest_candidates  

# First we update our do not harvest list 
do_not_harvest =[
    permno for permno, last_buy in last_buy_dates.items()
    if (day_2 - last_buy).days <= 30
]

harvest_threshold = -0.05
harvest_candidates =[]
dropped_more_than_threshold = 0

for permno, pos in positions.items():
    acb = pos['acb_per_share']
    price_today = prices_cad_2.get(permno, acb)
    
    return_pct = (price_today - acb) / acb
    
    if return_pct <= harvest_threshold:
        dropped_more_than_threshold += 1 
        #check if permno is allowed to be harvested 
        if permno not in do_not_harvest:
            harvest_candidates.append(permno)

print(f"Number of stocks down more than 5% today: {dropped_more_than_threshold}")
print(f"Number of stocks eligibale for Tax Loss Harvesting: {len(harvest_candidates)}")


Number of stocks down more than 5% today: 25
Number of stocks eligibale for Tax Loss Harvesting: 0


In [224]:
# FAST FORWARD: DAY 40 (lockouts are expired)
# ==========================================
# We jump ahead ~40 trading days so the CRA 30-day backward rule has expired.
day_40 = trading_days[40]
print(f"\n--- Fast Forwarding to Day 40: {day_40.date()} ---")

# 1. Get the new state of the world
daily_data_40 = df_univ.loc[day_40].copy()
daily_data_40 = daily_data_40.replace([np.inf, -np.inf], np.nan)

#current market prices for benchmark constituents 
prices_cad_40 = dict(zip(daily_data_40['permno'], daily_data_40['prc_adj_cad']))


# 2. Mark-to-Market
total_equity_cad_40 = 0.0
for permno, pos in positions.items():
    shares = pos['shares']
    price_today = prices_cad_40.get(permno)
    
    if price_today is None or pd.isna(price_today):
        price_today = pos['acb_per_share']
        
    total_equity_cad_40 += shares * price_today



--- Fast Forwarding to Day 40: 2015-03-03 ---


In [225]:
#========================
# DIVIDENDS 
divs_cad_40 = dict(zip(daily_data_40['permno'], daily_data_40['divamt_net_cad']))
# initialize day 40 dividends 
daily_dividend_income = 0.0
# Loop through what we own and see if they paid us today
for permno, pos in positions.items():
    dps = divs_cad_40.get(permno, 0.0)
    if dps > 0:
        payout = pos['shares'] * dps
        cash_cad += payout  # Add to our global cash balance!
        daily_dividend_income += payout

if daily_dividend_income > 0:
    print(f"Collected ${daily_dividend_income:,.2f} in CAD dividends on {day_40.date()}.")


total_aum = total_equity_cad_40 + cash_cad
print(f"Day 40 Total AUM: ${total_aum:,.4f}")

Collected $13.69 in CAD dividends on 2015-03-03.
Day 40 Total AUM: $1,087,099.1664


In [226]:
# 3. THE HARVEST SCANNER (With CRA Logic)
harvest_threshold = -0.05
harvest_candidates = []

# First we update our do not harvest list 
do_not_harvest = [
    permno for permno, last_buy in last_buy_dates.items()
    if (day_40 - last_buy).days <= 30
]

dropped_more_than_threshold = 0

for permno, pos in positions.items():
    acb = pos['acb_per_share']
    price_today = prices_cad_40.get(permno)
    
    if price_today is None or pd.isna(price_today) or acb <= 0:
        continue
        
    return_pct = (price_today - acb) / acb
    
    if return_pct <= harvest_threshold:
        dropped_more_than_threshold += 1 
        # The CRA Backward Rule Check
        if permno not in do_not_harvest :
            harvest_candidates.append(permno)
print(f"Number of stocks down > 5%: {dropped_more_than_threshold}")
print(f"Number of stocks down > 5% that are legally harvestable today: {len(harvest_candidates)}")
if len(harvest_candidates) > 0:
    print(f"Sample permno candidates to harvest: {harvest_candidates[:5]}")

Number of stocks down > 5%: 25
Number of stocks down > 5% that are legally harvestable today: 25
Sample permno candidates to harvest: [11618, 13936, 15553, 21776, 21792]


In [227]:
list(positions.items())[:5]

[(10104, {'shares': 198.7767, 'acb_per_share': 51.976925}),
 (10107, {'shares': 371.9906, 'acb_per_share': 54.826100000000004}),
 (10138, {'shares': 11.8628, 'acb_per_share': 100.55360000000002}),
 (10145, {'shares': 35.4341, 'acb_per_share': 117.51967500000002}),
 (10147, {'shares': 92.1108, 'acb_per_share': 34.975675})]

In [228]:
# DAY 40: PREPARING OPTIMIZER INPUTS
print("\n--- Prepping Data for the CVXPY Optimizer ---")

# 1. Get Day 40 Benchmark Weights
total_mkt_cap_40 = daily_data_40['mkt_cap_usd'].sum()
weights_40 = daily_data_40['mkt_cap_usd'] / total_mkt_cap_40
bench_w_40 = pd.Series(weights_40.values, index=daily_data_40['permno'])

# 2. Calculate our Current Portfolio Weights
current_w_40 = pd.Series({
    p: (pos['shares'] * prices_cad_40.get(p, pos['acb_per_share'])) / total_aum 
    for p, pos in positions.items()
})


--- Prepping Data for the CVXPY Optimizer ---


In [229]:
from b1_risk_model import FactorRiskModel
risk_model = FactorRiskModel()

# 3. Get the V Matrix (Union of Benchmark + Owned Stocks to prevent crashes)
owned_permnos = list(positions.keys())
benchmark_permnos_40 = daily_data_40['permno'].values
optimization_universe = np.unique(np.concatenate([benchmark_permnos_40, owned_permnos]))

V_mat_40 = risk_model.build_factor_covariance(day_40, optimization_universe)

# 4. Clean up any expired forward lockouts 
# Since we have not sold any positions, superficial_loss_lockouts is empty in this demo 
do_not_buy =[p for p, exp_date in superficial_loss_lockouts.items() if exp_date >= day_40]



Loading Risk Model inputs from Phase A (Quant Infrastructure)...
Building daily return matrix for Ledoit-Wolf failsafe...
Loading Factor Exposures (X)...
Loading Factor Covariance Matrices (F)...
Loading Idiosyncratic Risk (Delta)...


In [230]:
import cvxpy as cp
# ==========================================
# DAY 40: RUNNING THE OPTIMIZER
# ==========================================

# Align all vectors to the V_matrix index to ensure perfect linear algebra
permnos = V_mat_40.index.values
# Number of assets 
N = len(permnos)
# current portfolio and benchmark weights 
h_current = current_w_40.reindex(permnos).fillna(0.0).values
h_b = bench_w_40.reindex(permnos).fillna(0.0).values
V = V_mat_40.values
print('the risk model for the current period is ready!')

the risk model for the current period is ready!


In [231]:
# ==========================================
# BUILD THE OPPORTUNITY COST VECTOR
# ==========================================
# Initialize the penalty vector (Length N, matching V_matrix)
# The deeper the unrealized loss, the higher the opportunity cost (penalty) 
# of holding the asset and failing to exercise the tax-harvesting option.
harvest_opportunity_cost = np.zeros(N)

# for all the permnos in the V matrix, we are going to calculate the penalty vector 

# We only calculate penalties for the valid candidates we found earlier!
for i, permno in enumerate(permnos):
    if permno in harvest_candidates:
        acb = positions[permno]['acb_per_share']

        # get current price 
        price = prices_cad_40.get(permno, acb) 

        #calculate return from ACB
        return_pct = (price - acb) / acb
        
        # A -30% loss becomes a +0.30 opportunity cost.
        # CVXPY will minimize this, pushing the weight of this stock to 0.
        harvest_opportunity_cost[i] = abs(return_pct)

harvesting_candidates = pd.Series(harvest_opportunity_cost, index = permnos, name = 'harvest_oc')
harvesting_candidates = harvesting_candidates[harvesting_candidates != 0.0]

harvesting_candidates_permnos = harvesting_candidates.index
print(f"Opportunity cost vector is ready. There are {len(harvesting_candidates_permnos)}")
print(f"Here are the harvesting candidate permnos on {day_40.date()} \n {list(harvesting_candidates_permnos)}")

Opportunity cost vector is ready. There are 25
Here are the harvesting candidate permnos on 2015-03-03 
 [11618, 13936, 15553, 21776, 21792, 23026, 24010, 27828, 39538, 53613, 59176, 75100, 78877, 79089, 81774, 82298, 82618, 83435, 84032, 85072, 89004, 89641, 90071, 90162, 93159]


In [232]:
harvesting_candidates

11618    0.079296
13936    0.110544
15553    0.063145
21776    0.066762
21792    0.066233
23026    0.069270
24010    0.062013
27828    0.087784
39538    0.062575
53613    0.093698
59176    0.064988
75100    0.104839
78877    0.129468
79089    0.177134
81774    0.053593
82298    0.134961
82618    0.142006
83435    0.097960
84032    0.117327
85072    0.211715
89004    0.095661
89641    0.053968
90071    0.067903
90162    0.054049
93159    0.150392
Name: harvest_oc, dtype: float64

In [233]:
# setting the optimization hyperparams 

max_tracking_error=0.005 
# this translates to tracking error voaltility of sqrt(0.005) = 7.07%

turnover_penalty=0.01

# inititializing the optimization variable (portfolio weights)
h = cp.Variable(N)

# ==========================================
# BUILD THE OBJECTIVE AND CONSTRAINTS
# ==========================================        

# Opt. OBJECTIVE: Minimize (Holding Penalty + Tracking Error + Friction)

# Note: Since opportunity_cost_vector is positive for losers, 
# we MINIMIZE (h^T * opportunity_cost_vector).
# By minimizing a positive product, CVXPY forces h -> 0 for the losers.
tax_penalty = h.T @ harvest_opportunity_cost
turnover = cp.norm1(h - h_current)

objective = cp.Minimize(tax_penalty + (turnover_penalty * turnover))

#
# CONSTRAINTS
# 
constraints =[
    cp.sum(h) == 1.0, # fully invested
    h >= 0.0 # long only
]

# The double quadratic optimization: Tracking Error Leash
tracking_error = cp.quad_form(h - h_b, cp.psd_wrap(V))
constraints.append(tracking_error <= max_tracking_error)


# CRA Tax Rules (The 30-Day Lockouts)
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        # FORWARD RULE: We harvested this recently. Cannot increase weight.
        constraints.append(h[i] <= h_current[i])  
        
    if permno in do_not_harvest:
        # BACKWARD RULE: We bought this recently. Cannot sell at a loss.
        constraints.append(h[i] >= h_current[i])


In [234]:
import warnings
# ==========================================
# SOLVING THE OPTIMIZER
# ==========================================

prob = cp.Problem(objective, constraints)
try:
    prob.solve()
    if h.value is None:
        raise ValueError("Solver failed to find a feasible solution.")
    
    # Clip floating point noise and strictly re-normalize to 1.0 
    # to avoid shorting from computational jitter 
    h_opt = np.clip(h.value, 0.0, 1.0)
    h_opt =  pd.Series(h_opt / np.sum(h_opt), index=permnos)
    
except Exception as e:
    warnings.warn(f"CVXPY Solver Error: {e}. Falling back to Benchmark weights.")
    h_opt = pd.Series(h_b, index=permnos)


In [235]:
raw_delta = (h_opt - h_current)
raw_delta

10104    0.000053
10107    0.000043
10138    0.000046
10145    0.000056
10147    0.000056
           ...   
92988    0.000047
93002    0.000050
93096    0.000046
93159   -0.000293
93422    0.000066
Length: 504, dtype: float64

In [236]:
# increased positions 
raw_delta[raw_delta > 0 ]

10104    0.000053
10107    0.000043
10138    0.000046
10145    0.000056
10147    0.000056
           ...   
92890    0.000045
92988    0.000047
93002    0.000050
93096    0.000046
93422    0.000066
Length: 479, dtype: float64

In [237]:
# positions increased more than 1e-4
raw_delta[(raw_delta > 0) & (raw_delta > 1e-4)]

12622    0.000120
88436    0.000119
dtype: float64

Tracking Error Variance relies on an L2 quadratic norm, which heavily penalizes concentrated deviations. Consequently, the solver prefers to **'smear'** the proxy reinvestment across the entire benchmark to diffuse the variance penalty.

While mathematically correct, this is not tradable in real-life.  Paying a bid-ask spread on a $5 trade destroys the tax alpha we just generated.

In [238]:
# positions harvested and the weight harvested 
raw_delta[(raw_delta < 0)]

11618   -0.000630
13936   -0.000452
15553   -0.000299
21776   -0.001471
21792   -0.000463
23026   -0.000750
24010   -0.000725
27828   -0.003287
39538   -0.000471
53613   -0.001656
59176   -0.004393
75100   -0.000596
78877   -0.000559
79089   -0.000225
81774   -0.001131
82298   -0.000216
82618   -0.000885
83435   -0.002070
84032   -0.001215
85072   -0.000433
89004   -0.000480
89641   -0.001001
90071   -0.000423
90162   -0.000193
93159   -0.000293
dtype: float64

In [239]:
# the portoflio positions in the harvested permnos 
h_opt[raw_delta[(raw_delta < 0)].index]

11618    0.000000e+00
13936    0.000000e+00
15553    0.000000e+00
21776    0.000000e+00
21792    0.000000e+00
23026    0.000000e+00
24010    0.000000e+00
27828    0.000000e+00
39538    0.000000e+00
53613    0.000000e+00
59176    0.000000e+00
75100    0.000000e+00
78877    0.000000e+00
79089    0.000000e+00
81774    1.397846e-11
82298    0.000000e+00
82618    0.000000e+00
83435    0.000000e+00
84032    0.000000e+00
85072    0.000000e+00
89004    0.000000e+00
89641    6.855108e-12
90071    0.000000e+00
90162    2.857980e-11
93159    0.000000e+00
dtype: float64

### Two issues are observed
#### 1. No partial harvesting:

I used the L2 tracking error in the constraints to induce partial harvesting. The optimizer fully harvested the positions in the harvesting candidates. No partial harvesting. We might be able to force partial harvesting if we reduce the max_tracking_error of 0.005 which  translates to a significant tracking error voaltility of sqrt(0.005) = 7.07%. A lower MET would leash the optimizer. 

#### 2. L2 smear: 
Executing 400 microscopic trades destroys tax alpha through bid-ask friction. Also, for we might be able to partially address the L2 smear problem by increaasing the L1 norm.


In [240]:
import itertools 
import time 
# ==========================================
# THE HYPERPARAMETER GRID SEARCH
# ==========================================
print("\n--- Running Convex Optimization Grid Search ---")

# Define the Grid
tev_budgets =[1e-6, 5e-5, 5e-4, 5e-3]       # From ultra-tight to loose tracking error
turnover_penalties =[0.001, 0.01, 0.1, 1.0] # From cheap trading to extremely expensive

results =[]

# Define standard constraints that don't change
base_constraints =[
    cp.sum(h) == 1.0,
    h >= 0.0
]
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        base_constraints.append(h[i] <= h_current[i])
    if permno in do_not_harvest:
        base_constraints.append(h[i] >= h_current[i])

# the quadratic constraint is fixed in every loop 
risk_form = cp.quad_form(h - h_b, cp.psd_wrap(V))

# Run the sweep
total_runs = len(tev_budgets) * len(turnover_penalties)
run_count = 1

for tev, lam in itertools.product(tev_budgets, turnover_penalties):
    print(f"Solving {run_count}/{total_runs} | TEV: {tev} | Penalty: {lam}...", end="\r")
    
    # Objective: Minimize Tax Penalty + (Lambda * Turnover)
    tax_penalty = h.T @ harvest_opportunity_cost
    turnover = cp.norm1(h - h_current)
    objective = cp.Minimize(tax_penalty + (lam * turnover))
    
    # Constraints: Base + Specific TEV Leash
    constraints = base_constraints + [risk_form <= tev]
    
    prob = cp.Problem(objective, constraints)
    
    try:
        # Let CVXPY auto-route the QCQP problem
        prob.solve()
        
        if h.value is None:
            raise ValueError("Infeasible")
            
        h_opt = np.clip(h.value, 0.0, 1.0)
        h_opt = h_opt / np.sum(h_opt)
        
        # Calculate Actionable Metrics
        raw_delta = h_opt - h_current
        
        # Filter out the 1e-9 computational dust
        actionable_sells = (raw_delta < -1e-4).sum()
        actionable_buys = (raw_delta > 1e-4).sum()
        
        # Did it partially or fully harvest?
        max_harvest_depth = raw_delta.min()  # The biggest single sell
        max_proxy_concentration = raw_delta.max() # The biggest single proxy buy
        
        results.append({
            'TEV_Budget': tev,
            'Turnover_Penalty': lam,
            'Actionable_Sells': actionable_sells,
            'Actionable_Buys': actionable_buys,
            'Max_Sell_Weight': max_harvest_depth,
            'Max_Buy_Weight': max_proxy_concentration,
            'Status': prob.status
        })
        
    except Exception as e:
        results.append({
            'TEV_Budget': tev, 'Turnover_Penalty': lam, 
            'Actionable_Sells': 0, 'Actionable_Buys': 0, 
            'Max_Sell_Weight': 0, 'Max_Buy_Weight': 0, 'Status': 'Failed/Frozen'
        })
        
    run_count += 1

# Convert to DataFrame for beautiful viewing
df_results = pd.DataFrame(results)

print("\n\n--- GRID SEARCH RESULTS ---")
display(df_results.sort_values(['TEV_Budget', 'Turnover_Penalty']))

# Let's create a quick pivot table to show Buys (Proxy Smear vs Concentration)
print("\n--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---")
smear_matrix = df_results.pivot(index='TEV_Budget', columns='Turnover_Penalty', values='Actionable_Buys')
display(smear_matrix)


--- Running Convex Optimization Grid Search ---
Solving 16/16 | TEV: 0.005 | Penalty: 1.0.....

--- GRID SEARCH RESULTS ---


,TEV_Budget,Turnover_Penalty,Actionable_Sells,Actionable_Buys,Max_Sell_Weight,Max_Buy_Weight,Status
0,0.000001,0.001,25,2,-4.393470e-03,1.008924e-04,optimal
1,0.000001,0.010,25,2,-4.393470e-03,1.201143e-04,optimal
2,0.000001,0.100,1,0,-4.328196e-04,2.099593e-06,optimal
3,0.000001,1.000,0,0,3.192375e-09,4.116019e-07,optimal
4,0.000050,0.001,25,2,-4.393470e-03,1.009200e-04,optimal
5,0.000050,0.010,25,2,-4.393470e-03,1.196308e-04,optimal
6,0.000050,0.100,1,0,-4.328194e-04,2.098366e-06,optimal
7,0.000050,1.000,0,0,3.087744e-09,4.108717e-07,optimal
8,0.000500,0.001,25,2,-4.393470e-03,1.007649e-04,optimal
9,0.000500,0.010,25,2,-4.393470e-03,1.196102e-04,optimal



--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---


Turnover_Penalty,0.001,0.010,0.100,1.000
TEV_Budget,,,,
0.000001,2,2,0,0
0.000050,2,2,0,0
0.000500,2,2,0,0
0.005000,2,2,0,0


In [241]:

# ==========================================
# THE HYPERPARAMETER GRID SEARCH
# ==========================================
print("\n--- Running Convex Optimization Grid Search ---")

# Define the Grid
tev_budgets =[1e-9, 5e-9, 1e-8, 1e-6]       # From ultra-tight to loose tracking error
turnover_penalties =[0.07, 0.8, 0.9, 0.1, 0.11, 0.12, 0.13] # From cheap trading to extremely expensive

results =[]

# Define standard constraints that don't change
base_constraints =[
    cp.sum(h) == 1.0,
    h >= 0.0
]
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        base_constraints.append(h[i] <= h_current[i])
    if permno in do_not_harvest:
        base_constraints.append(h[i] >= h_current[i])

# the quadratic constraint is fixed in every loop 
risk_form = cp.quad_form(h - h_b, cp.psd_wrap(V))

# Run the sweep
total_runs = len(tev_budgets) * len(turnover_penalties)
run_count = 1

for tev, lam in itertools.product(tev_budgets, turnover_penalties):
    print(f"Solving {run_count}/{total_runs} | TEV: {tev} | Penalty: {lam}...", end="\r")
    
    # Objective: Minimize Tax Penalty + (Lambda * Turnover)
    tax_penalty = h.T @ harvest_opportunity_cost
    turnover = cp.norm1(h - h_current)
    objective = cp.Minimize(tax_penalty + (lam * turnover))
    
    # Constraints: Base + Specific TEV Leash
    constraints = base_constraints + [risk_form <= tev]
    
    prob = cp.Problem(objective, constraints)
    
    try:
        # Let CVXPY auto-route the QCQP problem
        prob.solve()
        
        if h.value is None:
            raise ValueError("Infeasible")
            
        h_opt = np.clip(h.value, 0.0, 1.0)
        h_opt = h_opt / np.sum(h_opt)
        
        # Calculate Actionable Metrics
        raw_delta = h_opt - h_current
        
        # Filter out the 1e-9 computational dust
        actionable_sells = (raw_delta < -1e-4).sum()
        actionable_buys = (raw_delta > 1e-4).sum()
        
        # Did it partially or fully harvest?
        max_harvest_depth = raw_delta.min()  # The biggest single sell
        max_proxy_concentration = raw_delta.max() # The biggest single proxy buy
        
        results.append({
            'TEV_Budget': tev,
            'Turnover_Penalty': lam,
            'Actionable_Sells': actionable_sells,
            'Actionable_Buys': actionable_buys,
            'Max_Sell_Weight': max_harvest_depth,
            'Max_Buy_Weight': max_proxy_concentration,
            'Status': prob.status
        })
        
    except Exception as e:
        results.append({
            'TEV_Budget': tev, 'Turnover_Penalty': lam, 
            'Actionable_Sells': 0, 'Actionable_Buys': 0, 
            'Max_Sell_Weight': 0, 'Max_Buy_Weight': 0, 'Status': 'Failed/Frozen'
        })
        
    run_count += 1

# Convert to DataFrame for beautiful viewing
df_results = pd.DataFrame(results)

print("\n\n--- GRID SEARCH RESULTS ---")
display(df_results.sort_values(['TEV_Budget', 'Turnover_Penalty']))

# Let's create a quick pivot table to show Buys (Proxy Smear vs Concentration)
print("\n--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---")
smear_matrix = df_results.pivot(index='TEV_Budget', columns='Turnover_Penalty', values='Actionable_Buys')
display(smear_matrix)


--- Running Convex Optimization Grid Search ---
Solving 28/28 | TEV: 1e-06 | Penalty: 0.13...

--- GRID SEARCH RESULTS ---


,TEV_Budget,Turnover_Penalty,Actionable_Sells,Actionable_Buys,Max_Sell_Weight,Max_Buy_Weight,Status
0,1.000000e-09,0.07,6,6,-2.239914e-03,1.620300e-03,optimal
3,1.000000e-09,0.10,7,6,-2.237703e-03,1.593421e-03,optimal
4,1.000000e-09,0.11,7,6,-2.238112e-03,1.595002e-03,optimal
5,1.000000e-09,0.12,7,6,-2.237538e-03,1.589523e-03,optimal
6,1.000000e-09,0.13,6,6,-2.237653e-03,1.588445e-03,optimal
1,1.000000e-09,0.80,5,5,-2.236978e-03,1.581417e-03,optimal
2,1.000000e-09,0.90,5,5,-2.236973e-03,1.581468e-03,optimal
7,5.000000e-09,0.07,5,4,-2.185193e-03,1.332650e-03,optimal
10,5.000000e-09,0.10,5,4,-2.191026e-03,1.281349e-03,optimal
11,5.000000e-09,0.11,5,4,-2.186966e-03,1.268150e-03,optimal



--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---


Turnover_Penalty,0.07,0.10,0.11,0.12,0.13,0.80,0.90
TEV_Budget,,,,,,,
1.000000e-09,6,6,6,6,6,5,5
5.000000e-09,4,4,4,4,3,3,3
1.000000e-08,4,3,3,3,3,3,3
1.000000e-06,0,0,0,0,0,0,0


In [242]:
# setting the optimization hyperparams 

max_tracking_error= 5e-9

turnover_penalty= 0.10

# inititializing the optimization variable (portfolio weights)
h = cp.Variable(N)

# ==========================================
# BUILD THE OBJECTIVE AND CONSTRAINTS
# ==========================================        

# Opt. OBJECTIVE: Minimize (Holding Penalty + Tracking Error + Friction)

# Note: Since opportunity_cost_vector is positive for losers, 
# we MINIMIZE (h^T * opportunity_cost_vector).
# By minimizing a positive product, CVXPY forces h -> 0 for the losers.
tax_penalty = h.T @ harvest_opportunity_cost
turnover = cp.norm1(h - h_current)

objective = cp.Minimize(tax_penalty + (turnover_penalty * turnover))

#
# CONSTRAINTS
# 
constraints =[
    cp.sum(h) == 1.0, # fully invested
    h >= 0.0 # long only
]

# The double quadratic optimization: Tracking Error Leash
tracking_error = cp.quad_form(h - h_b, cp.psd_wrap(V))
constraints.append(tracking_error <= max_tracking_error)


# CRA Tax Rules (The 30-Day Lockouts)
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        # FORWARD RULE: We harvested this recently. Cannot increase weight.
        constraints.append(h[i] <= h_current[i])  
        
    if permno in do_not_harvest:
        # BACKWARD RULE: We bought this recently. Cannot sell at a loss.
        constraints.append(h[i] >= h_current[i])


In [243]:
# ==========================================
# SOLVING THE OPTIMIZER
# ==========================================

prob = cp.Problem(objective, constraints)
try:
    prob.solve()
    if h.value is None:
        raise ValueError("Solver failed to find a feasible solution.")
    
    # Clip floating point noise and strictly re-normalize to 1.0 
    # to avoid shorting from computational jitter 
    h_opt = np.clip(h.value, 0.0, 1.0)
    h_opt =  pd.Series(h_opt / np.sum(h_opt), index=permnos)
    
except Exception as e:
    warnings.warn(f"CVXPY Solver Error: {e}. Falling back to Benchmark weights.")
    h_opt = pd.Series(h_b, index=permnos)


In [244]:
raw_delta = (h_opt - h_current)
raw_delta

10104   -9.683777e-09
10107   -1.139499e-08
10138   -2.718862e-08
10145   -4.868783e-08
10147   -1.536587e-08
             ...     
92988    1.851193e-08
93002    2.495960e-08
93096   -2.953648e-09
93159   -2.944716e-05
93422    1.829249e-09
Length: 504, dtype: float64

In [245]:
# increased positions 
raw_delta[raw_delta > 0 ]

10299    6.282541e-10
10909    7.800655e-09
11552    2.439026e-10
11600    3.960897e-09
11762    1.146983e-08
             ...     
92293    7.986784e-09
92778    7.125702e-09
92988    1.851193e-08
93002    2.495960e-08
93422    1.829249e-09
Length: 128, dtype: float64

In [246]:
# Actionable BUYS
# positions increased more than 1e-4
raw_delta[(raw_delta > 0) & (raw_delta > 1e-4)]

12622    0.001281
60097    0.001271
69550    0.000137
88436    0.000686
dtype: float64

In [247]:
# positions harvested and the weight harvested 
raw_delta[(raw_delta < 0)]

10104   -9.683777e-09
10107   -1.139499e-08
10138   -2.718862e-08
10145   -4.868783e-08
10147   -1.536587e-08
             ...     
92655   -5.658256e-08
92709   -3.852412e-08
92890   -2.838281e-08
93096   -2.953648e-09
93159   -2.944716e-05
Length: 376, dtype: float64

In [248]:
# Actionable Sells
# positions harvested 
raw_delta[(raw_delta < 0) & (raw_delta < -1e-4)]

76149   -0.000121
79089   -0.000117
84032   -0.000119
85072   -0.000325
92156   -0.002191
dtype: float64

In [249]:
# Did we partially harvest? 
# the portoflio positions in the harvested permnos 
h_opt[raw_delta[(raw_delta < 0) & (raw_delta < -1e-4)].index]

76149    0.000275
79089    0.000108
84032    0.001096
85072    0.000108
92156    0.000065
dtype: float64

In [250]:
any(h_opt < 1e-6)

False

In [254]:
# ==========================================
# updating the portfolio positions based on opt. outputs h_opt
# translating Weights -> Target Shares
# ==========================================
print("\n--- Translating Weights to Target Shares ---")
MIN_TRADE_DOLLARS = 50.0  # Setting the minimum trade size (e.g. $50 CAD)


# initialize a dictionary for all the positions in the optimization output
target_shares = {}

for permno, weight in h_opt.items():
    if permno in prices_cad_40:
        price = prices_cad_40[permno]
        tgt_dollars = weight * total_aum
        
        # What do we currently own in dollars?
        cur_shares = positions.get(permno, {}).get('shares', 0.0)
        cur_dollars = cur_shares * price
        
        # Calculate the absolute dollar value of the proposed trade
        trade_dollar_value = abs(tgt_dollars - cur_dollars)
        
        # THE DUST FILTER: Only accept trades larger than $50
        if trade_dollar_value >= MIN_TRADE_DOLLARS:
            raw_shares = tgt_dollars / price
            target_shares[permno] = np.floor(raw_shares * 10000.0) / 10000.0
        else:
            # It's dust! Keep the target shares equal to current shares
            target_shares[permno] = cur_shares

print("Dust Filter Applied. Microscopic trades eliminated.")


--- Translating Weights to Target Shares ---
Dust Filter Applied. Microscopic trades eliminated.


In [255]:
realized_gains_cad = 0.0
realized_losses_cad = 0.0

# ==========================================
# Process SELLS
# ==========================================
print("\n--- Executing Sells (Funding Cash & Harvesting) ---")
harvested_count = 0

# We iterate over a list of keys to delete from the dict if shares hit 0
for permno in list(positions.keys()):
    tgt_shares = target_shares.get(permno, 0.0)
    cur_shares = positions[permno]['shares']
    delta = tgt_shares - cur_shares
    
    # If delta is negative, the optimizer told us to SELL
    if delta < -1e-6: # first dust filter 
        # safety check to not sell more than we own (floating point safety)
        shares_to_sell = min(abs(delta), cur_shares)
        
        acb = positions[permno]['acb_per_share']
        # if permno is dropped from the benchmark, just update with acb (will fix later)
        price = prices_cad_40.get(permno, acb)

        # 1. Update Cash Balance from cash generated from sells
        cash_cad += (shares_to_sell * price)
        
        # 2. Calculate Realized PnL for Tax Tracking
        realized_pnl = (price - acb) * shares_to_sell
        
        if realized_pnl < -1e-4:  # It's a loss!
            realized_losses_cad += abs(realized_pnl)
            
            #===================================
            # APPLY THE CRA FORWARD RULE
            #===================================
            # We harvested a loss, so we are locked out of buying this for 30 days
            superficial_loss_lockouts[permno] = day_40 + pd.Timedelta(days=30)
            harvested_count += 1
            print(f"HARVESTED: Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Loss: -${abs(realized_pnl):,.2f}")
            
        else:  # It's a gain
            realized_gains_cad += realized_pnl
            print(f"SOLD (Gain): Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Gain: +${realized_pnl:,.2f}")
            
        # 3. Deduct shares from our ledger
        positions[permno]['shares'] -= shares_to_sell
        
        # 4. Clean up empty positions if we sold the whole position
        if positions[permno]['shares'] < 1e-6:
            del positions[permno]

print(f"\n--- End of Sell Execution ---")
print(f"Successfully harvested {harvested_count} positions.")
print(f"Updated Cash Balance ready for Buys: ${cash_cad:,.2f}")


--- Executing Sells (Funding Cash & Harvesting) ---
HARVESTED: Permno 13936 | Sold 0.4130 shrs | Loss: -$6.76
HARVESTED: Permno 59176 | Sold 0.9733 shrs | Loss: -$6.90
SOLD (Gain): Permno 66800 | Sold 0.7621 shrs | Gain: +$2.60
SOLD (Gain): Permno 76149 | Sold 10.4336 shrs | Gain: +$0.00
HARVESTED: Permno 78877 | Sold 4.6235 shrs | Loss: -$13.87
HARVESTED: Permno 79089 | Sold 1.1939 shrs | Loss: -$27.36
HARVESTED: Permno 84032 | Sold 1.9131 shrs | Loss: -$17.24
HARVESTED: Permno 85072 | Sold 2.0868 shrs | Loss: -$94.85
SOLD (Gain): Permno 92156 | Sold 20.4954 shrs | Gain: +$0.00

--- End of Sell Execution ---
Successfully harvested 6 positions.
Updated Cash Balance ready for Buys: $3,866.67


In [256]:
# ==========================================
# Process BUYS
# ==========================================
print("\n--- Executing Buys (Proxy Reinvestment) ---")
bought_count = 0

for permno, tgt_shares in target_shares.items():
    # using double get to prevent crash if we are buying brand new stocks (e.g. in the benchmark but not in positions)
    cur_shares = positions.get(permno, {}).get('shares', 0.0)
    delta = tgt_shares - cur_shares
    
    # If delta is positive, the optimizer told us to BUY
    if delta > 1e-6:
        # Failsafe 1: Ensure we aren't legally locked out from buying (Forward Rule)
        if permno in superficial_loss_lockouts:
            print(f"WARNING: Optimizer tried to buy {permno} but it is locked out! Skipping.")
            continue
            
        price = prices_cad_40[permno]
        cost = delta * price
        
        # Failsafe 2: Ensure we have enough cash (Floating point/Margin protection)
        if cash_cad >= cost:
            cash_cad -= cost
        else:
            # If rounding dust caused a slight cash overdraft, buy exactly what we can afford
            affordable_shares = np.floor((cash_cad / price) * 10000.0) / 10000.0
            if affordable_shares <= 0: continue
            delta = affordable_shares
            cost = delta * price
            cash_cad -= cost

        # 1. CRA ACB Pooling Math
        if permno not in positions: # if we are initiating a position in a stock:
            positions[permno] = {'shares': delta, 'acb_per_share': price}
        else: 
            # if permno exists in the positions, we just update the number of shares and the acb
            old_shares = positions[permno]['shares']
            old_acb = positions[permno]['acb_per_share']
            new_shares = old_shares + delta
            
            # (Old Total Cost + New Total Cost) / New Total Shares
            new_acb = ((old_shares * old_acb) + cost) / new_shares
            positions[permno] = {'shares': new_shares, 'acb_per_share': new_acb}
            
        # Log the Backward Rule
        # If we buy this today, we cannot harvest a loss on it for the next 30 days.
        last_buy_dates[permno] = day_40
        bought_count += 1
        
        print(f"BOUGHT: Permno {permno} | Bought {delta:,.4f} shrs | Cost: ${cost:,.2f}")

print(f"\n--- End of Buy Execution ---")
print(f"Successfully reinvested cash into {bought_count} proxy positions.")
print(f"Final End of Day Cash Balance: ${cash_cad:,.2f}")


--- Executing Buys (Proxy Reinvestment) ---
BOUGHT: Permno 12622 | Bought 15.8050 shrs | Cost: $1,392.95
BOUGHT: Permno 60097 | Bought 14.2236 shrs | Cost: $1,381.79
BOUGHT: Permno 69550 | Bought 2.1571 shrs | Cost: $148.88
BOUGHT: Permno 88436 | Bought 6.9798 shrs | Cost: $746.28

--- End of Buy Execution ---
Successfully reinvested cash into 4 proxy positions.
Final End of Day Cash Balance: $196.77


In [192]:
# ==========================================
# Process BUYS
# ==========================================
print("\n--- Executing Buys (Proxy Reinvestment) ---")
bought_count = 0

for permno, tgt_shares in target_shares.items():
    # using double get to prevent crash if we are buying brand new stocks (e.g. in the benchmark but not in positions)
    cur_shares = positions.get(permno, {}).get('shares', 0.0)
    delta = tgt_shares - cur_shares
    
    # If delta is positive, the optimizer told us to BUY
    if delta > 1e-6:
        # Failsafe 1: Ensure we aren't legally locked out from buying (Forward Rule)
        if permno in superficial_loss_lockouts:
            print(f"WARNING: Optimizer tried to buy {permno} but it is locked out! Skipping.")
            continue
            
        price = prices_cad_40[permno]
        cost = delta * price
        
        # Failsafe 2: Ensure we have enough cash (Floating point/Margin protection)
        if cash_cad >= cost:
            cash_cad -= cost
        else:
            # If rounding dust caused a slight cash overdraft, buy exactly what we can afford
            affordable_shares = np.floor((cash_cad / price) * 10000.0) / 10000.0
            if affordable_shares <= 0: continue
            delta = affordable_shares
            cost = delta * price
            cash_cad -= cost

        # 1. CRA ACB Pooling Math
        if permno not in positions: # if we are initiating a position in a stock:
            positions[permno] = {'shares': delta, 'acb_per_share': price}
        else: 
            # if permno exists in the positions, we just update the number of shares and the acb
            old_shares = positions[permno]['shares']
            old_acb = positions[permno]['acb_per_share']
            new_shares = old_shares + delta
            
            # (Old Total Cost + New Total Cost) / New Total Shares
            new_acb = ((old_shares * old_acb) + cost) / new_shares
            positions[permno] = {'shares': new_shares, 'acb_per_share': new_acb}
            
        # Log the Backward Rule
        # If we buy this today, we cannot harvest a loss on it for the next 30 days.
        last_buy_dates[permno] = day_40
        bought_count += 1
        
        print(f"BOUGHT: Permno {permno} | Bought {delta:,.4f} shrs | Cost: ${cost:,.2f}")

print(f"\n--- End of Buy Execution ---")
print(f"Successfully reinvested cash into {bought_count} proxy positions.")
print(f"Final End of Day Cash Balance: ${cash_cad:,.2f}")


--- Executing Buys (Proxy Reinvestment) ---
BOUGHT: Permno 11762 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 12062 | Bought 0.0001 shrs | Cost: $0.02
BOUGHT: Permno 12622 | Bought 15.8050 shrs | Cost: $1,392.95
BOUGHT: Permno 13168 | Bought 0.0003 shrs | Cost: $0.03
BOUGHT: Permno 13407 | Bought 0.1044 shrs | Cost: $10.35
BOUGHT: Permno 13721 | Bought 0.0002 shrs | Cost: $0.01
BOUGHT: Permno 13788 | Bought 0.0002 shrs | Cost: $0.01
BOUGHT: Permno 14011 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 14297 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 19583 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 21207 | Bought 0.0007 shrs | Cost: $0.02
BOUGHT: Permno 23501 | Bought 0.0001 shrs | Cost: $0.00
BOUGHT: Permno 25785 | Bought 0.0020 shrs | Cost: $0.04
BOUGHT: Permno 27887 | Bought 0.0002 shrs | Cost: $0.02
BOUGHT: Permno 29102 | Bought 0.0007 shrs | Cost: $0.01
BOUGHT: Permno 38156 | Bought 0.0003 shrs | Cost: $0.02
BOUGHT: Permno 39917 | Bought 0.0002 shrs | Cost: $0.

In [ ]:

print("\n--- Translating Weights to Target Shares ---")
# initialize a dictionary for all the positions in the optimization output
target_shares = {}

for permno, weight in h_opt.items():
    # Only process meaningful weights and stocks we actually have prices for
    if weight > 1e-6 and permno in prices_cad_40:
        target_dollars = weight * total_aum
        raw_shares = target_dollars / prices_cad_40[permno]
        
        # Truncate to 4 decimals for Wealthsimple fractional shares
        # Flooring prevents microscopic rounding that could cause negative cash
        target_shares[permno] = np.floor(raw_shares * 10000.0) / 10000.0


--- Translating Weights to Target Shares ---


In [ ]:
realized_gains_cad = 0.0
realized_losses_cad = 0.0

# ==========================================
# Process SELLS
# ==========================================
print("\n--- Executing Sells (Funding Cash & Harvesting) ---")
harvested_count = 0

# We iterate over a list of keys to delete from the dict if shares hit 0
for permno in list(positions.keys()):
    tgt_shares = target_shares.get(permno, 0.0)
    cur_shares = positions[permno]['shares']
    delta = tgt_shares - cur_shares
    
    # If delta is negative, the optimizer told us to SELL
    if delta < -1e-6: # first dust filter 
        # safety check to not sell more than we own (floating point safety)
        shares_to_sell = min(abs(delta), cur_shares)
        
        acb = positions[permno]['acb_per_share']
        # if permno is dropped from the benchmark, just update with acb (will fix later)
        price = prices_cad_40.get(permno, acb)

        # 1. Update Cash Balance from cash generated from sells
        cash_cad += (shares_to_sell * price)
        
        # 2. Calculate Realized PnL for Tax Tracking
        realized_pnl = (price - acb) * shares_to_sell
        
        if realized_pnl < -1e-4:  # It's a loss!
            realized_losses_cad += abs(realized_pnl)
            
            #===================================
            # APPLY THE CRA FORWARD RULE
            #===================================
            # We harvested a loss, so we are locked out of buying this for 30 days
            superficial_loss_lockouts[permno] = day_40 + pd.Timedelta(days=30)
            harvested_count += 1
            print(f"HARVESTED: Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Loss: -${abs(realized_pnl):,.2f}")
            
        else:  # It's a gain
            realized_gains_cad += realized_pnl
            print(f"SOLD (Gain): Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Gain: +${realized_pnl:,.2f}")
            
        # 3. Deduct shares from our ledger
        positions[permno]['shares'] -= shares_to_sell
        
        # 4. Clean up empty positions if we sold the whole position
        if positions[permno]['shares'] < 1e-6:
            del positions[permno]

print(f"\n--- End of Sell Execution ---")
print(f"Successfully harvested {harvested_count} positions.")
print(f"Updated Cash Balance ready for Buys: ${cash_cad:,.2f}")


--- Executing Sells (Funding Cash & Harvesting) ---
SOLD (Gain): Permno 10104 | Sold 0.0002 shrs | Gain: +$0.00
HARVESTED: Permno 10107 | Sold 0.0003 shrs | Loss: -$0.00
SOLD (Gain): Permno 10138 | Sold 0.0003 shrs | Gain: +$0.00
SOLD (Gain): Permno 10145 | Sold 0.0005 shrs | Gain: +$0.01
SOLD (Gain): Permno 10147 | Sold 0.0005 shrs | Gain: +$0.00
HARVESTED: Permno 10516 | Sold 0.0006 shrs | Loss: -$0.00
SOLD (Gain): Permno 10696 | Sold 0.0002 shrs | Gain: +$0.00
SOLD (Gain): Permno 11308 | Sold 0.0006 shrs | Gain: +$0.00
SOLD (Gain): Permno 11404 | Sold 0.0002 shrs | Gain: +$-0.00
HARVESTED: Permno 11618 | Sold 0.3901 shrs | Loss: -$1.72
HARVESTED: Permno 11674 | Sold 0.0002 shrs | Loss: -$0.00
SOLD (Gain): Permno 11703 | Sold 0.0003 shrs | Gain: +$0.00
SOLD (Gain): Permno 11850 | Sold 0.0013 shrs | Gain: +$0.00
SOLD (Gain): Permno 11955 | Sold 0.0009 shrs | Gain: +$0.01
SOLD (Gain): Permno 12052 | Sold 0.0003 shrs | Gain: +$0.00
SOLD (Gain): Permno 12073 | Sold 0.0030 shrs | Gain: +

### The Incidental Wash Sale Trap.
1. Why 56 Harvests instead of 25?
We correctly identified 25 candidates that crossed the strict -5% threshold. We assigned those 25 a heavy Tax Penalty in the objective function, so the optimizer (partially) sold them as instructed.
Where did the other 31 come from?
**The "L2 Proxy Smear"** The optimizer is making microscopic trades (e.g., selling 0.0003 shares) across the portfolio to re-align Tracking Error.
**We have incidental harvest!** If the optimizer decides to shave 0.0003 shares off a stock that happens to be down -2% (which didn't trigger the scanner), the Ledger records that the price is lower than the ACB.
Ledger logs a capital loss of $0.00 and locks that stock out for 30 days.

**This is not my desired output** We triggered a 30-day CRA forward lockout on 31 stocks for literally zero pennies of tax benefit. If one of those 31 stocks crashes by 20% tomorrow, we are legally blocked from harvesting it! NOT GOOD!

2. The Solution: **The Minimum Trade Filter (Squashing the Dust)**
We need to tell the Execution Layer: "If the dollar value of the trade is less than $50, do not execute it. It is just optimization dust."
